**Installing Unisloth**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch
major_version, minor_version = torch.cuda.get_device_capability()

# 1. Install Unsloth automatically based on GPU
if major_version >= 8:
    !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    !pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install ALL dependencies at once (including zoo, trl, and hf_transfer)
!pip install --no-deps xformers trl peft accelerate bitsandbytes unsloth_zoo
!pip install hf_transfer

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ksnnx9cj/unsloth_50dbe50e0b8f477d89a2d0761f34abcf
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ksnnx9cj/unsloth_50dbe50e0b8f477d89a2d0761f34abcf
  Resolved https://github.com/unslothai/unsloth.git to commit 4328d0b4f691f30a0c645d2b3d9d7c421599b914
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for unsloth: filename=unsloth-2026.4.4-py3-none-any.whl size=29650898 sha256=3199c3bb9db25deee07fc187b4f3ffd8b3d6b08a2e4c0bd87650e154afff5ccd
  Stored in directory: /tmp/pip-ephem-wheel-cache-z4b9om3g/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 54.3 MB/s eta 0:00:00
   ━━━━━━

In [1]:
import os

# Enable resilient downloading
os.environ["HTTP_TIMEOUT"] = "300"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Clear old broken downloads
!rm -rf /root/.cache/huggingface/hub/models--unsloth--llama-3-8b-bnb-4bit

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/llama-3-8b-bnb-4bit",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
        force_download = False,
    )
    print("✅ Model loaded successfully!")
except Exception as e:
    print(f"❌ Network error: {e}")

/tmp/ipykernel_3191/1182652364.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


✅ Model loaded successfully!


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ LoRA Expert Layer Attached and Ready for Phase 3.")

Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ LoRA Expert Layer Attached and Ready for Phase 3.


In [4]:
from datasets import load_dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []

    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)

    return { "text" : texts, }

dataset_path = "/content/drive/MyDrive/Sentinel_Project/dataset.jsonl"
dataset = load_dataset("json", data_files=dataset_path, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1717 [00:00<?, ? examples/s]

In [6]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.9 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.4.6 requires cut_cross_entropy; python_version >= "3.10", which is not installed.
unsloth-zoo 2026.4.6 requires msgspec, which is not installed.
unsloth-zoo 2026.4.6 requires tyro, which is not installed.
unsloth-zoo 2026.4.6 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.8.4 which is incompatible.
uns

In [7]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# We now use SFTConfig as the primary source of truth for the Unsloth 2026 kernels
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    # We move the length and packing settings into the Config to stop the ValueError
    args = SFTConfig(
        max_seq_length = max_seq_length, # Pass it here
        packing = True,                  # Pass it here too!
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        push_to_hub = False,
    ),
)

# START THE ENGINE
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1717 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/1717 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,576 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.855787
2,0.648517
3,0.733287
4,0.784966
5,0.531515
6,0.739659
7,0.746028
8,0.859305
9,0.324803
10,0.518410


In [8]:
# 1. Define the save path in your Google Drive
save_path = "/content/drive/MyDrive/Sentinel_Project/Llama3_Sentinel_v1"

# 2. Save the trained LoRA adapters
model.save_pretrained(save_path)

# 3. Save the tokenizer (so it remembers how to read Solidity code)
tokenizer.save_pretrained(save_path)

print(f"✅ Success! Your trained auditor is safely stored at: {save_path}")

✅ Success! Your trained auditor is safely stored at: /content/drive/MyDrive/Sentinel_Project/Llama3_Sentinel_v1
